<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Air-Quality-_PM-2.5/target_PM_2_5_Statistical_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.api import VAR

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load your datasets
train_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/PM 2.5/pm25_training_dataset.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/PM 2.5/pm25_testing_dataset.csv")

target = "air_quality_PM2.5"

# ⚠️ If you have datetime column, set it (optional but recommended)
# train_df["datetime"] = pd.to_datetime(train_df["datetime"])
# test_df["datetime"] = pd.to_datetime(test_df["datetime"])

# train_df.set_index("datetime", inplace=True)
# test_df.set_index("datetime", inplace=True)

# Evaluation function
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true_safe = np.where(np.array(y_true) == 0, 1e-10, y_true)
    acc = 100 - (np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100)

    return mse, rmse, mae, r2, acc

# **ARIMA**

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load datasets
train_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/PM 2.5/pm25_training_dataset.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/PM 2.5/pm25_testing_dataset.csv")

target = "air_quality_PM2.5"

# Keep only target and make numeric
train_target = pd.to_numeric(train_df[target], errors="coerce").dropna()
test_target = pd.to_numeric(test_df[target], errors="coerce").dropna()

print("Train target length:", len(train_target))
print("Test target length:", len(test_target))

# Evaluation function
def evaluate(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true_safe = np.where(y_true == 0, 1e-10, y_true)
    accuracy = 100 - (np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100)

    return mse, rmse, mae, r2, accuracy

# Fit ARIMA
model = ARIMA(train_target, order=(5, 1, 0))
model_fit = model.fit()

# Forecast exactly test length
forecast = model_fit.forecast(steps=len(test_target))

print("Forecast length:", len(forecast))

# Make sure both lengths match
min_len = min(len(test_target), len(forecast))
test_target = test_target.iloc[:min_len]
forecast = np.array(forecast)[:min_len]

metrics_arima = evaluate(test_target, forecast)

print("\nARIMA Results")
print("MSE:", metrics_arima[0])
print("RMSE:", metrics_arima[1])
print("MAE:", metrics_arima[2])
print("R2:", metrics_arima[3])
print("Accuracy (%):", metrics_arima[4])

Train target length: 104002
Test target length: 26001
Forecast length: 26001

ARIMA Results
MSE: 604.4042173230242
RMSE: 24.584633764264705
MAE: 14.147890800360747
R2: -0.00952263924358121
Accuracy (%): -35.866786230739535


# **SARIMA**

In [ ]:
model = SARIMAX(
    train_df[target],
    order=(2,1,2),
    seasonal_order=(1,1,1,12)  # adjust if needed
)

model_fit = model.fit(disp=False)

forecast = model_fit.forecast(steps=len(test_df))

metrics_sarima = evaluate(test_df[target], forecast)

print("SARIMA Results")
print(metrics_sarima)

SARIMA Results
(603.6057706222608, np.float64(24.568389662781335), 14.581010139055785, -0.008189011850608718, np.float64(-47.473190121647775))


In [ ]:
print("SARIMA Results")

print("MSE:", metrics_sarima[0])
print("RMSE:", metrics_sarima[1])
print("MAE:", metrics_sarima[2])
print("R2:", metrics_sarima[3])
print("Accuracy (%):", metrics_sarima[4])

SARIMA Results
MSE: 603.6057706222608
RMSE: 24.568389662781335
MAE: 14.581010139055785
R2: -0.008189011850608718
Accuracy (%): -47.473190121647775


# **VAR**

In [ ]:
var_features = [
    "air_quality_PM2.5",
    "air_quality_PM10",
    "air_quality_Carbon_Monoxide",
    "air_quality_Nitrogen_dioxide",
    "temperature_celsius",
    "humidity"
]

train_var = train_df[var_features].dropna()
test_var = test_df[var_features].dropna()

model = VAR(train_var)
model_fit = model.fit(maxlags=5)

lag_order = model_fit.k_ar

forecast = model_fit.forecast(
    train_var.values[-lag_order:],
    steps=len(test_var)
)

forecast_df = pd.DataFrame(
    forecast,
    index=test_var.index,
    columns=var_features
)

metrics_var = evaluate(
    test_var["air_quality_PM2.5"],
    forecast_df["air_quality_PM2.5"]
)

print("VAR Results")
print(metrics_var)

VAR Results
(627.8661009040632, np.float64(25.057256452055224), 17.361395855571544, -0.048710490611099555, np.float64(-112.86974201994147))


In [ ]:
results = pd.DataFrame([
    {
        "Model": "ARIMA",
        "MSE": metrics_arima[0],
        "RMSE": metrics_arima[1],
        "MAE": metrics_arima[2],
        "R2": metrics_arima[3],
        "Accuracy (%)": metrics_arima[4]
    },
    {
        "Model": "SARIMA",
        "MSE": metrics_sarima[0],
        "RMSE": metrics_sarima[1],
        "MAE": metrics_sarima[2],
        "R2": metrics_sarima[3],
        "Accuracy (%)": metrics_sarima[4]
    },
    {
        "Model": "VAR",
        "MSE": metrics_var[0],
        "RMSE": metrics_var[1],
        "MAE": metrics_var[2],
        "R2": metrics_var[3],
        "Accuracy (%)": metrics_var[4]
    }
]).round(4)

print(results)

    Model       MSE     RMSE      MAE      R2  Accuracy (%)
0   ARIMA  604.4042  24.5846  14.1479 -0.0095      -35.8668
1  SARIMA  603.6058  24.5684  14.5810 -0.0082      -47.4732
2     VAR  627.8661  25.0573  17.3614 -0.0487     -112.8697


# **Test Results**
ARIMA
SARIMA
VAR

In [ ]:
import pandas as pd

stat_results_table = pd.DataFrame([
    {
        "Model": "ARIMA",
        "MSE": metrics_arima[0],
        "RMSE": metrics_arima[1],
        "MAE": metrics_arima[2],
        "R2": metrics_arima[3],
        "Accuracy (%)": metrics_arima[4]
    },
    {
        "Model": "SARIMA",
        "MSE": metrics_sarima[0],
        "RMSE": metrics_sarima[1],
        "MAE": metrics_sarima[2],
        "R2": metrics_sarima[3],
        "Accuracy (%)": metrics_sarima[4]
    },
    {
        "Model": "VAR",
        "MSE": metrics_var[0],
        "RMSE": metrics_var[1],
        "MAE": metrics_var[2],
        "R2": metrics_var[3],
        "Accuracy (%)": metrics_var[4]
    }
]).round(4)

print(stat_results_table)

    Model       MSE     RMSE      MAE      R2  Accuracy (%)
0   ARIMA  604.4042  24.5846  14.1479 -0.0095      -35.8668
1  SARIMA  603.6058  24.5684  14.5810 -0.0082      -47.4732
2     VAR  627.8661  25.0573  17.3614 -0.0487     -112.8697


In [ ]:
stat_results_table.to_csv("statistical_model_results.csv", index=False)

print("Statistical results saved as CSV!")

Statistical results saved as CSV!


In [ ]:
from google.colab import files
files.download("statistical_model_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>